# Reproduction script: PTSD symptom detection from trauma narratives
Run from the repository root. Sections marked HAS_TEXT need the unshared narratives and are skipped on the public release file.
Changes against the previously uploaded script are marked CHANGED / NEW in comments.

In [ ]:
# ---- SETUP ----
import os
import sys
import pickle
import numpy as np
import pandas as pd
if HAS_TEXT:
    import spacy
    nlp = spacy.load("de_core_news_lg")
from scipy.stats import spearmanr, ttest_ind

sys.path.append(os.getcwd())  # CHANGED: no hardcoded working directory, run from repo root

# CHANGED: removed "from src.functions import bootstrap_ci" (it triggers model loading at import and is not used)
from src.classifier_function import (
    make_pipelines,
    run_nested_cv,
    classifier_builder_multi,
    make_pipelines_ngram,
    pick_best_family,
    permutation_test_nested_cv,
    participant_level_single_ci,       # CHANGED: typo "particiapnt" fixed
    participant_level_paired_bootstrap,  # CHANGED: was never imported
    report_precision_recall,
)

DATA_PATH = "data_for_public/dataset_minimal_release.csv"  # CHANGED: release file
RUN_PERMUTATIONS = False   # NEW: set True to rerun the slow permutation tests (80 s per permutation)
N_PERMUTATIONS = 500

dataset = pd.read_csv(DATA_PATH, sep=";", decimal=".", encoding="utf-8")

target_col = "condition_pcl"
positive_class = "clin_pcl"

# NEW: narratives are not part of the public release
HAS_TEXT = "response_trauma" in dataset.columns
print("Narrative text available:", HAS_TEXT)

needed = [target_col, "PCL-5_total"]
missing = [c for c in needed if c not in dataset.columns]
assert not missing, f"Missing columns in release file: {missing}"

## 1. Narrative conditions (trauma / negative / neutral), single run reused everywhere

In [ ]:
condition_results, complete_dataset, embedding_cols_by_condition, y_data, pipelines_list = classifier_builder_multi(
    dataset=dataset,
    target_col=target_col,
    positive_class=positive_class,
    response_types=("response_trauma", "response_negnt", "response_neutr"),
)

### 1b. Verify identical outer-fold membership (needed for participant-level paired comparisons)

In [ ]:
def _folds(key):
    # CHANGED: dummy X, only fold membership matters
    return list(condition_results[key]["outer_cv"].split(np.zeros(len(y_data)), y_data))

trauma_folds, negnt_folds, neutr_folds = _folds("response_trauma"), _folds("response_negnt"), _folds("response_neutr")
fold_check_passed = True
for i in range(5):
    m1 = np.array_equal(trauma_folds[i][1], negnt_folds[i][1])
    m2 = np.array_equal(trauma_folds[i][1], neutr_folds[i][1])
    print(f"Fold {i}: trauma vs negnt = {m1}, trauma vs neutr = {m2}")
    fold_check_passed &= m1 and m2
assert fold_check_passed, "Outer fold membership differs across conditions."
print("Confirmed: outer fold membership identical across conditions.")

## 2. TF-IDF n-gram baseline (3 families, 40 configurations). Needs narrative text.

In [ ]:
ngram_results = None
if HAS_TEXT:
    assert len(make_pipelines_ngram()) == 40, "make_pipelines_ngram must be the 40-config three-family grid"  # NEW
    X_text_trauma = complete_dataset["response_trauma"].astype(str).to_numpy()
    ngram_results = run_nested_cv(X_text_trauma, y_data, make_pipelines_ngram(), n_outer_splits=5, n_inner_splits=5)
else:
    print("Skipped (no narrative text). n-gram scores are loaded from the released out-of-fold file in section 16.")

## 3. Subjective-measures classifier

In [ ]:
subjective_feature_cols = [
    f"{p}_trauma_1" for p in
    ["vivid", "emoArous", "valence", "detail", "relevance", "imagery", "perspec", "active", "ease"]
]
sub_dataset = dataset.dropna(subset=subjective_feature_cols + [target_col]).reset_index(drop=True)
X_subjective = sub_dataset[subjective_feature_cols].to_numpy()
y_subjective = ((sub_dataset[target_col] == positive_class) * 1).to_numpy()
subjective_results = run_nested_cv(X_subjective, y_subjective, make_pipelines(), n_outer_splits=5, n_inner_splits=5)

## 4. PCL-5 / DASS-21 Spearman correlations

In [ ]:
pcl5_col = "PCL-5_total"
dass_cols = {
    "DASS-21 total": "dass21_total",
    "DASS-21 depression": "DASS21_depression",
    "DASS-21 anxiety": "DASS21_anxiety",
    "DASS-21 stress": "DASS21_stress",
}
correlations_spearman = {}
for label, col in dass_cols.items():
    sub = dataset[[pcl5_col, col]].dropna()
    r, p = spearmanr(sub[pcl5_col], sub[col])
    print(f"{pcl5_col} vs {col}: rho={r:.3f}, p={p:.4f}, n={len(sub)}")
    correlations_spearman[label] = (r, p)

## 5. DASS-21 subscale classifiers

In [ ]:
dass_targets = {
    "DASS21_depression": ("condition_dass_depression", "clin_dass_depression"),
    "DASS21_anxiety": ("condition_dass_anxiety", "clin_dass_anxiety"),
    "DASS21_stress": ("condition_dass_stress", "clin_dass_stress"),
}
embedding_cols_trauma = embedding_cols_by_condition["response_trauma"]

dass_results = {}
for subscale, (tcol, pclass) in dass_targets.items():
    sub_ds = dataset.dropna(subset=embedding_cols_trauma + [tcol]).reset_index(drop=True)
    X = sub_ds[embedding_cols_trauma].to_numpy()
    y = ((sub_ds[tcol] == pclass) * 1).to_numpy()
    dass_results[subscale] = run_nested_cv(X, y, make_pipelines(), n_outer_splits=5, n_inner_splits=5)

## 6. DSM-5 symptom clusters (median split)

In [ ]:
cluster_items = {
    "Intrusion": [f"PCL-5_{i}" for i in range(1, 6)],
    "Avoidance": [f"PCL-5_{i}" for i in range(6, 8)],
    "NegCognitionsMood": [f"PCL-5_{i}" for i in range(8, 15)],
    "ArousalReactivity": [f"PCL-5_{i}" for i in range(15, 21)],
}
for cluster_name, items in cluster_items.items():
    dataset[f"PCL5_{cluster_name}"] = dataset[items].sum(axis=1)
    median_val = dataset[f"PCL5_{cluster_name}"].median()
    dataset.loc[dataset[f"PCL5_{cluster_name}"] > median_val, f"condition_{cluster_name}"] = f"high_{cluster_name}"
    dataset.loc[dataset[f"PCL5_{cluster_name}"] <= median_val, f"condition_{cluster_name}"] = f"low_{cluster_name}"

cluster_targets = {c: (f"condition_{c}", f"high_{c}") for c in cluster_items}
cluster_results = {}
for cluster, (tcol, pclass) in cluster_targets.items():
    sub_ds = dataset.dropna(subset=embedding_cols_trauma + [tcol]).reset_index(drop=True)
    X = sub_ds[embedding_cols_trauma].to_numpy()
    y = ((sub_ds[tcol] == pclass) * 1).to_numpy()
    cluster_results[cluster] = run_nested_cv(X, y, make_pipelines(), n_outer_splits=5, n_inner_splits=5)

## 7. PCL-5 cutoff sensitivity

In [ ]:
cutoff_results = {}
for cutoff in (31, 33, 34, 36, 38):
    sub_ds = dataset.dropna(subset=embedding_cols_trauma + [pcl5_col]).reset_index(drop=True)
    X = sub_ds[embedding_cols_trauma].to_numpy()
    y = (sub_ds[pcl5_col] >= cutoff).astype(int).to_numpy()
    cutoff_results[cutoff] = run_nested_cv(X, y, make_pipelines(), n_outer_splits=5, n_inner_splits=5, verbose=False)

## 8. First-person subject analysis (spaCy). Parsing needs narrative text; the t-test runs from the derived column.

In [ ]:
fp_col = "response_trauma_fp_subject_rate"

if HAS_TEXT:
    nlp = spacy.load("de_core_news_lg")
    FIRST_PERSON_FORMS = {
        "ich", "mich", "mir", "wir", "uns",
        "mein", "meine", "meiner", "meinem", "meinen", "meines",
        "unser", "unsere", "unserer", "unserem", "unseren", "unseres",
    }

    def extract_first_person_features(text):
        doc = nlp(text)
        n_tokens = len([t for t in doc if not t.is_punct and not t.is_space])
        fp_subject_count = sum(1 for t in doc if t.text.lower() in FIRST_PERSON_FORMS and t.dep_ in ("sb", "nsubj"))
        return {"n_tokens": n_tokens, "fp_subject_count": fp_subject_count,
                "fp_subject_rate": fp_subject_count / n_tokens if n_tokens > 0 else np.nan}

    for i in range(len(dataset)):
        text = str(dataset.at[i, "response_trauma"]).strip()
        feats = extract_first_person_features(text) if text else {"n_tokens": np.nan, "fp_subject_count": np.nan, "fp_subject_rate": np.nan}
        for k, v in feats.items():
            dataset.loc[i, f"response_trauma_{k}"] = v

fp_t = fp_p = np.nan
if fp_col in dataset.columns:  # CHANGED: runs on the release file as well
    high = dataset.loc[dataset[target_col] == positive_class, fp_col].dropna()
    low = dataset.loc[dataset[target_col] != positive_class, fp_col].dropna()
    fp_t, fp_p = ttest_ind(high, low, equal_var=False)
    print(f"First-person subject rate, high vs low PCL-5: t={fp_t:.2f}, p={fp_p:.3f}")
    print(f"Group means: high={high.mean():.4f}, low={low.mean():.4f}")  # NEW: direction
else:
    print(f"Column {fp_col} not in file, skipped.")

## 9. Permutation tests (slow). Restricted to the family with the highest mean outer-fold AUC.

In [ ]:
perm_results = {}
if RUN_PERMUTATIONS:
    jobs = [
        ("Trauma", condition_results["response_trauma"], complete_dataset[embedding_cols_by_condition["response_trauma"]].to_numpy(), y_data, pipelines_list),
        ("Negative", condition_results["response_negnt"], complete_dataset[embedding_cols_by_condition["response_negnt"]].to_numpy(), y_data, pipelines_list),
        ("Neutral", condition_results["response_neutr"], complete_dataset[embedding_cols_by_condition["response_neutr"]].to_numpy(), y_data, pipelines_list),
    ]
    if HAS_TEXT and ngram_results is not None:
        jobs.append(("N-gram", ngram_results, X_text_trauma, y_data, make_pipelines_ngram()))  # CHANGED: new 40-config grid
    for label, results, X, y, pls in jobs:
        fam, _ = pick_best_family(results)
        fam_pipelines = [p for p in pls if p[0].startswith(fam)]  # prefix match: ngram families are named ngram_logreg etc.
        assert len(fam_pipelines) > 0, f"No pipelines for family {fam}"  # NEW: prevents the empty-list bug
        obs, perm_scores, pval = permutation_test_nested_cv(X, y, fam_pipelines, n_permutations=N_PERMUTATIONS)
        perm_results[label] = {"observed": obs, "pvalue": pval, "family": fam}

    for subscale, (tcol, pclass) in dass_targets.items():
        sub_ds = dataset.dropna(subset=embedding_cols_trauma + [tcol]).reset_index(drop=True)
        X = sub_ds[embedding_cols_trauma].to_numpy()
        y = ((sub_ds[tcol] == pclass) * 1).to_numpy()
        fam, _ = pick_best_family(dass_results[subscale])
        fam_pipelines = [p for p in make_pipelines() if p[0] == fam]
        obs, perm_scores, pval = permutation_test_nested_cv(X, y, fam_pipelines, n_permutations=N_PERMUTATIONS)
        perm_results[f"DASS {subscale}"] = {"observed": obs, "pvalue": pval, "family": fam}

    for cluster, (tcol, pclass) in cluster_targets.items():
        sub_ds = dataset.dropna(subset=embedding_cols_trauma + [tcol]).reset_index(drop=True)
        X = sub_ds[embedding_cols_trauma].to_numpy()
        y = ((sub_ds[tcol] == pclass) * 1).to_numpy()
        fam, _ = pick_best_family(cluster_results[cluster])
        fam_pipelines = [p for p in make_pipelines() if p[0] == fam]
        obs, perm_scores, pval = permutation_test_nested_cv(X, y, fam_pipelines, n_permutations=N_PERMUTATIONS)
        perm_results[f"Cluster {cluster}"] = {"observed": obs, "pvalue": pval, "family": fam}
    for k, v in perm_results.items():
        print(k, v)
else:
    print("RUN_PERMUTATIONS = False, skipped.")

## 10. Final metrics (pooled AUC with participant-level CI, mean outer-fold AUC, precision/recall/F1/accuracy)

In [ ]:
all_analyses = {
    "Trauma": condition_results["response_trauma"],
    "Negative": condition_results["response_negnt"],
    "Neutral": condition_results["response_neutr"],
    **({"N-gram baseline": ngram_results} if ngram_results is not None else {}),
    "Subjective measures": subjective_results,
    **{f"DASS {k}": v for k, v in dass_results.items()},
    **{f"Cluster {k}": v for k, v in cluster_results.items()},
}

final_metrics = {}
for label, results in all_analyses.items():
    fam, _ = pick_best_family(results)
    pooled = results["per_family_pooled"][fam]
    pooled_auc, ci_low, ci_high = participant_level_single_ci(pooled["y_true"], pooled["y_score"])
    mean_fold_auc = results["per_family_auc"][fam].mean()
    threshold = 0.0 if fam.endswith("svm") else 0.5
    precision, recall, f1, acc = report_precision_recall(pooled, threshold=threshold, label=label)
    final_metrics[label] = {
        "family": fam, "pooled_auc": pooled_auc, "ci": (ci_low, ci_high),
        "mean_fold_auc": mean_fold_auc,
        "precision": precision, "recall": recall, "f1": f1, "accuracy": acc,
    }
    print(f"{label}: family={fam}, pooled AUC={pooled_auc:.3f} [{ci_low:.3f}, {ci_high:.3f}], mean outer-fold AUC={mean_fold_auc:.3f}")

## 11. Pairwise comparisons (CHANGED: moved after section 10, which creates all_analyses and final_metrics)

In [ ]:
def compare(label_a, label_b):
    pa = all_analyses[label_a]["per_family_pooled"][final_metrics[label_a]["family"]]
    pb = all_analyses[label_b]["per_family_pooled"][final_metrics[label_b]["family"]]
    assert np.array_equal(pa["participant_idx"], pb["participant_idx"])
    diff, lo, hi, _ = participant_level_paired_bootstrap(pa["y_true"], pa["y_score"], pb["y_score"])
    print(f"{label_a} vs {label_b}: diff={diff:.3f}, 95% CI=[{lo:.3f}, {hi:.3f}]")
    return diff, lo, hi

pairwise_comparisons = {
    "Trauma_vs_Negative": compare("Trauma", "Negative"),
    "Trauma_vs_Neutral": compare("Trauma", "Neutral"),
    "Trauma_vs_Subjective": compare("Trauma", "Subjective measures"),
}
if "N-gram baseline" in all_analyses:
    pairwise_comparisons["Trauma_vs_Ngram"] = compare("Trauma", "N-gram baseline")

## 12. Primary versus inner-selected estimates (Supplementary Table 3)
Primary: best family chosen by mean outer-fold AUC. Inner-selected: model chosen per fold by the inner loop only.

In [ ]:
supp_table3 = {}
for label, results in all_analyses.items():
    fam, _ = pick_best_family(results)
    pooled = results["per_family_pooled"][fam]
    p_auc, p_lo, p_hi = participant_level_single_ci(pooled["y_true"], pooled["y_score"])
    i_auc, i_lo, i_hi = participant_level_single_ci(results["pooled_y_true"], results["pooled_y_score"])
    mean_outer = float(np.mean(results["outer_auc_scores"]))
    supp_table3[label] = {"primary": (p_auc, p_lo, p_hi), "inner_selected": (i_auc, i_lo, i_hi), "mean_outer": mean_outer}
    print(f"{label:28s} primary {p_auc:.3f} [{p_lo:.3f}, {p_hi:.3f}] | inner-selected {i_auc:.3f} [{i_lo:.3f}, {i_hi:.3f}] | mean outer {mean_outer:.3f}")

## 13. Model families (trauma) and n-gram family table (Supplementary Table 4)

In [ ]:
for label in ["Trauma"] + (["N-gram baseline"] if "N-gram baseline" in all_analyses else []):
    res = all_analyses[label]
    print(label)
    for fam, v in res["per_family_auc"].items():
        pooled = res["per_family_pooled"][fam]
        a, lo, hi = participant_level_single_ci(pooled["y_true"], pooled["y_score"])
        print(f"  {fam:14s} outer-fold AUCs {np.round(v, 3)} mean={v.mean():.3f} SD={v.std():.3f} | pooled {a:.3f} [{lo:.3f}, {hi:.3f}]")

## 14. Narrative length statistics (needs text)

In [ ]:
if HAS_TEXT:
    from scipy.stats import ttest_ind as _tt
    for col in ["response_trauma", "response_negnt", "response_neutr"]:
        wc = dataset[col].astype(str).str.split().str.len()
        hi_ = wc[dataset[target_col] == positive_class]
        lo_ = wc[dataset[target_col] != positive_class]
        tw, pw = _tt(hi_, lo_, equal_var=False)
        ts, ps = _tt(hi_, lo_, equal_var=True)
        print(f"{col}: mean={wc.mean():.2f}, SD={wc.std():.2f}, min={wc.min()}, max={wc.max()} | Welch t={tw:.3f}, p={pw:.3f} | Student t={ts:.3f}, p={ps:.3f}")
else:
    print("Skipped (no narrative text).")

## 15. Save participant-level out-of-fold scores (no narrative text)

In [ ]:
if "N-gram baseline" in all_analyses:
    t_fam = final_metrics["Trauma"]["family"]
    n_fam = final_metrics["N-gram baseline"]["family"]
    pt = all_analyses["Trauma"]["per_family_pooled"][t_fam]
    pn = all_analyses["N-gram baseline"]["per_family_pooled"][n_fam]
    assert np.array_equal(pt["participant_idx"], pn["participant_idx"])
    assert np.array_equal(pt["y_true"], pn["y_true"])
    oof = pd.DataFrame({
        "participant_idx": pt["participant_idx"],
        "y_true": pt["y_true"],
        "score_embeddings_" + t_fam: pt["y_score"],
        "score_ngram_" + n_fam: pn["y_score"],
    })
    os.makedirs("data_for_public", exist_ok=True)
    oof.to_csv("data_for_public/oof_scores_embeddings_vs_ngram.csv", index=False)
    print("Saved data_for_public/oof_scores_embeddings_vs_ngram.csv", oof.shape)
else:
    print("Skipped (n-gram run needs narrative text).")

## 16. Embeddings vs n-gram comparison from the released scores alone (runs without text)

In [ ]:
oof = pd.read_csv("data_for_public/oof_scores_embeddings_vs_ngram.csv")
score_cols = [c for c in oof.columns if c.startswith("score_")]
emb_col = [c for c in score_cols if "embeddings" in c][0]
ng_col = [c for c in score_cols if "ngram" in c][0]
diff, lo, hi, _ = participant_level_paired_bootstrap(oof["y_true"].to_numpy(), oof[emb_col].to_numpy(), oof[ng_col].to_numpy())
print(f"Embeddings vs n-gram: diff={diff:.3f}, 95% CI=[{lo:.3f}, {hi:.3f}]")

## 17. Save everything

In [ ]:
with open("full_results_final.pkl", "wb") as f:
    pickle.dump({
        "final_metrics": final_metrics,
        "perm_results": perm_results,
        "pairwise_comparisons": pairwise_comparisons,
        "correlations_spearman": correlations_spearman,
        "supp_table3": supp_table3,
        "cutoff_results_summary": {
            c: participant_level_single_ci(
                cutoff_results[c]["per_family_pooled"][pick_best_family(cutoff_results[c])[0]]["y_true"],
                cutoff_results[c]["per_family_pooled"][pick_best_family(cutoff_results[c])[0]]["y_score"],
            ) for c in cutoff_results
        },
        "first_person_test": (fp_t, fp_p),
        "selection_paths": {k: {"pooled_y_true": v["pooled_y_true"], "pooled_y_score": v["pooled_y_score"],
                                "outer_auc_scores": v["outer_auc_scores"]} for k, v in all_analyses.items()},
    }, f)
print("Saved full_results_final.pkl")

## Optional: merge the three spaCy columns into the release file (run once with the full data, then re-export)

In [ ]:
# fp = pd.read_csv("data/fp_features_trauma.csv")   # contains ResponseId, which the release file drops
# assert len(fp) == len(dataset) == 150
# for c in ["response_trauma_n_tokens", "response_trauma_fp_subject_count", "response_trauma_fp_subject_rate"]:
#     dataset[c] = fp[c].to_numpy()   # positional merge, row order must match
# dataset.to_csv(DATA_PATH, sep=";", index=False, encoding="utf-8")